In [7]:
from __future__ import annotations

import sys
from pathlib import Path

try:
    import pandas as pd
except Exception:
    pd = None

cwd = Path.cwd().resolve()
if (cwd / "quantum_config.json").exists():
    REPO_ROOT = cwd
elif (cwd.parent / "quantum_config.json").exists():
    REPO_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Cannot find quantum_config.json. Start notebook from repo root or notebooks directory."
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from quantum.runtime_ops import list_account_remaining

rows = list_account_remaining(REPO_ROOT / "quantum_config.json")
print(f"Account count: {len(rows)}")

if pd is not None:
    from IPython.display import display

    display(pd.DataFrame(rows))
else:
    for row in rows:
        print(row)


Account count: 4


,index,name,remaining_seconds,remaining_minutes,channel,instance,backend,error
0,1,jiyuan wang,329.0,5.483333,ibm_cloud,research,ibm_fez,None
1,3,jiyuan wang gmail,328.0,5.466667,ibm_cloud,research,ibm_fez,None
2,2,sen he utsa,279.0,4.650000,ibm_cloud,research,ibm_fez,None
3,0,sen he,216.0,3.600000,ibm_cloud,research,ibm_fez,None


In [8]:
from quantum.runtime_ops import list_jobs_for_account, cancel_job, create_runtime_client_for_account

ACCOUNT_NAME = "sen he"  # Fill account name, e.g. "sen he"
DRY_RUN = True      # Keep True for preview; set False to actually cancel
RECENT_LIMIT = 5    # Only check the most recent 5 jobs

if not ACCOUNT_NAME:
    raise ValueError("Please set ACCOUNT_NAME first")

jobs = list_jobs_for_account(
    account_name=ACCOUNT_NAME,
    config_path=REPO_ROOT / "quantum_config.json",
    limit=RECENT_LIMIT,
    pending=None,
)

pending_states = {"INITIALIZING", "QUEUED", "RUNNING", "VALIDATING"}
targets = [j for j in jobs if str(j.get("status") or "UNKNOWN") in pending_states and j.get("job_id")]

print(f"Recent jobs for account {ACCOUNT_NAME}: {len(jobs)}")
print(f"Pending jobs among them: {len(targets)}")

results = []
if DRY_RUN:
    for j in targets:
        s = str(j.get("status") or "UNKNOWN")
        results.append({
            "job_id": str(j["job_id"]),
            "status_before": s,
            "cancel_called": False,
            "status_after": s,
        })
else:
    client = create_runtime_client_for_account(
        account_name=ACCOUNT_NAME,
        config_path=REPO_ROOT / "quantum_config.json",
    )
    for j in targets:
        results.append(
            cancel_job(
                client=client,
                job_id=str(j["job_id"]),
                dry_run=False,
                wait_seconds=1.0,
            )
        )

print(f"Cancellation result count for account {ACCOUNT_NAME}: {len(results)}")
if pd is not None and results:
    from IPython.display import display
    display(pd.DataFrame(results))
else:
    for row in results:
        print(row)


Recent jobs for account sen he: 5
Pending jobs among them: 0
Cancellation result count for account sen he: 0
